In [2]:
import os
path_data = "/Users/cristianb/Documents/Python/rel8ed/SynapseIQ_Lab_01/Data/ohio"
os.makedirs(path_data, exist_ok=True)

In [4]:
def split_driver_name(name: str):
    """
    Splits a driver's full name into [first_name, middle_name, last_name] 
    following specific rules about hyphens and suffixes like Jr., Sr., III, etc.
    """
    if not isinstance(name, str):
        return [None, None, None]

    name = name.strip()

    if name.lower() == 'unknown' or ('unknown' in name.lower() and len(name) < 10):
        return [None, None, None]

    suffixes = {"sr.", "jr.", "ii", "iii", "iv", "v", "jr"}
    parts = name.split()

    # Normalize all parts to lower for suffix detection
    parts_lower = [p.lower() for p in parts]

    # Rule 3: Handle suffixes (Sr., Jr., III, etc.)
    for i, part in enumerate(parts_lower):
        if part in suffixes and i > 0:
            last_name = parts[i-1]
            parts = parts[:i-1]  # Remove last name and suffix from parts
            break
    else:
        last_name = None

    # Rebuild name after removing suffix, if found
    name_cleaned = ' '.join(parts)

    # Rule 1: Check for "-"
    if "-" in name_cleaned:
        parts = name_cleaned.split()
        # Find the part containing "-"
        for idx, p in enumerate(parts):
            if "-" in p:
                last_name = p
                if ',' not in name_cleaned and len(parts) >= 4:
                    # Rule 2: if 4+ words and no comma
                    middle_name = parts[idx - 1] if idx >= 1 else None
                    first_name = ' '.join(parts[:idx-1]) if idx >= 2 else None
                else:
                    # Default split for hyphen without 4+ words
                    if ',' in name_cleaned:
                        # Format: Last, First Middle
                        last, first_middle = [part.strip() for part in name_cleaned.split(',', 1)]
                        parts = first_middle.split()
                        first_name = parts[0] if len(parts) >= 1 else None
                        middle_name = ' '.join(parts[1:]) if len(parts) > 1 else None
                    else:
                        # Assume last name and rest
                        parts = name_cleaned.split()
                        last_name = p
                        parts.remove(p)
                        first_name = parts[0] if len(parts) >= 1 else None
                        middle_name = ' '.join(parts[1:]) if len(parts) > 1 else None
                break
    else:
        if ',' in name_cleaned:
            # Format: Last, First Middle
            last, first_middle = [part.strip() for part in name_cleaned.split(',', 1)]
            parts = first_middle.split()
            first_name = parts[0] if len(parts) >= 1 else None
            middle_name = ' '.join(parts[1:]) if len(parts) > 1 else None
            last_name = last if not last_name else last_name
        else:
            # Default format: First Middle Last
            parts = name_cleaned.split()
            if len(parts)==2 and not last_name:
                last_name = parts[-1] if len(parts) >= 1 else None
                parts = parts[:-1]
            if not last_name and len(parts)==3:
                last_name = parts[-1] if len(parts) >= 1 else None
                parts = parts[:-1]
            if not last_name and len(parts)>3:
                last_name = parts[-2] if len(parts) >= 1 else None
                parts = parts[:-1]
                parts = parts[:-1]
            first_name = parts[0] if len(parts) >= 1 else None
            middle_name = ' '.join(parts[1:]) if len(parts) > 1 else None

    # Normalize names (capitalize properly)
    def normalize(n):
        return n.title() if isinstance(n, str) else None

    return [normalize(first_name), normalize(middle_name), normalize(last_name)]

In [14]:
import pandas as pd
import psycopg2
from datetime import datetime

def extract_city_from_address(address: str):
    if not isinstance(address, str):
        return None
    parts = [part.strip() for part in address.split(",")]
    if len(parts) == 4:
        return parts[1]  # segundo componente: ciudad
    return None


def extract_year(date_str):
    try:
        if not isinstance(date_str, str):
            return None
        cleaned = date_str.strip().lower()
        if cleaned in ("null", "n/a", "none", ""):
            return None
        return datetime.strptime(cleaned, "%m/%d/%Y").year
    except (ValueError, TypeError):
        return None


def clean_str(value):
    if pd.isna(value) or str(value).strip().lower() in {"null", "", "none", "N/A"}:
        return None
    return str(value).strip()

def to_int(value):
    try:
        if isinstance(value, str) and value.strip().lower() in ("n/a", "na", "none", ""):
            return None
        return int(float(value))
    except (ValueError, TypeError):
        return None

def to_datetime(value):
    try:
        return pd.to_datetime(value)
    except:
        return None

def insert_data_from_dataframe(df: pd.DataFrame, path_files_saved):
    conn = psycopg2.connect(
        dbname="crash_records_001",
        user="synapseiq",
        password="SynapseIQ$2025",
        host="localhost",
        port="5433"
    )
    cur = conn.cursor()

    for _, row in df.iterrows():
        #print(row)
        report_number = clean_str(row.get("Accident Report Number"))
        crash_date = to_datetime(row.get("Crash Date"))
        city =  extract_city_from_address(clean_str(row.get("Address")))  
        street = clean_str(row.get("Address"))
        type_ps = clean_str(row.get("TYPE"))
        unit_fault = clean_str(row.get("Unit at Fault"))
        generation_date = datetime.now()
        state = "OH"
        name = clean_str(row.get("Name"))
        first, middle, last = split_driver_name(name)
        gender = clean_str(row.get("Gender"))
        phone1 = clean_str(row.get("Contacts_1"))
        age = to_int(row.get("Age"))
        year_birth = extract_year(row.get("Birth"))
        passenger_notes = f"Minors: {clean_str(row.get('Minors'))}"
        license_plate = clean_str(row.get("License Plate"))
        unit_number = to_int(row.get("Unit"))  
        insurance_company = clean_str(row.get("Unit at Fault Company"))
        policy_number = clean_str(row.get("Unit at Fault Policy"))
        notes = f"Unit at Fault: {clean_str(row.get('Unit at Fault'))}; " \
                f"Insurance Policy: {clean_str(row.get('Insurance Policy'))}; " \
                f"Insurance Company: {clean_str(row.get('Insurance Company'))}; "


        # Verificar incidente por report_number y fecha
        cur.execute("""
            SELECT id FROM incident_reports 
            WHERE report_number = %s AND accident_datetime = %s
        """, (report_number, crash_date))
        incident = cur.fetchone()

        # Si no exoste registro
        if incident:
            incident_id = incident[0]
        else:
            cur.execute("""
                INSERT INTO incident_reports (
                    report_number, accident_datetime, 
                    city, street, generation_date, 
                    state, internal_report_number, 
                    original_document_location
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                RETURNING id
            """, (report_number, crash_date, 
                  city, street, generation_date, 
                  state, "oh"+str(report_number), 
                  os.path.join(path_files_saved,report_number + ".pdf")))
            incident_id = cur.fetchone()[0]

        # Verificar vehículo
        cur.execute("""
            SELECT id FROM vehicles 
            WHERE incident_report_id = %s AND license_plate_number = %s AND unit_number = %s
        """, (incident_id, license_plate, unit_number))
        vehicle = cur.fetchone()
        if vehicle:
            vehicle_id = vehicle[0]
        else:
            cur.execute("""
                INSERT INTO vehicles (
                    incident_report_id, unit_number, license_plate_number, 
                    insurance_company, policy_number, notes,
                    driver_name, driver_first_name, driver_middle_name, driver_last_name
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s) RETURNING id
            """, (incident_id, unit_number, license_plate, 
                  insurance_company, policy_number, notes, 
                  name, first, middle, last))
            vehicle_id = cur.fetchone()[0]

    
        # Verificar pasajero
        cur.execute("""
            SELECT id FROM passengers
            WHERE vehicle_id = %s AND name = %s AND gender = %s
        """, (vehicle_id, name, gender))
        passenger = cur.fetchone()
        if not passenger:
            cur.execute("""
                INSERT INTO passengers (
                    vehicle_id, name, gender, 
                    phone1, age, year_birth, notes,
                    first_name, middle_name, last_name, 
                    state, city, street, role
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """, (vehicle_id, name, gender, 
                  phone1, age, year_birth, passenger_notes, 
                  first, middle, last, 
                  state, city, street, type_ps))

    conn.commit()
    cur.close()
    conn.close()


In [4]:
from seleniumbase import SB
import time
from datetime import datetime, timedelta
import requests

username = 'rrel8ed'
password = 'zFfUPRWH6q'
country = 'US'
entry = ('http://customer-%s-cc-%s:%s@pr.oxylabs.io:7777' %
    (username, country, password))
 
proxy='customer-%s-cc-%s:%s@pr.oxylabs.io:7777' %(username, country, password)

# 1 day ago
past_day = datetime.now() - timedelta(2)
# Format 1: 2024.11.6 (NOV)
past_day = past_day.strftime('%m.%d.%Y')

# start browser with SeleniumBase
with SB(uc=True,proxy=proxy) as sb:
    # Open website
    sb.driver.maximize_window()
    sb.open("https://ohtrafficdata.dps.ohio.gov/CrashRetrieval")
    

    search_box = sb.driver.find_element("xpath", '//*[@id="txtCrashStartDate"]') 
    search_box.send_keys(past_day)

    search_button = sb.driver.find_element("xpath", '//*[@id="ddlAgencies"]') 
    search_button = sb.driver.find_element("xpath", '//*[@id="ddlAgencies"]/option[@value="661"]') 
    search_button.click()

    # Blue Ash = 375
    # Cheviot = 376
    # Fairfield = 85
    # Amelia = 128
    # HAMILTON = 86     
    # INDIAN HILL = 388
    # LEBANON = 934
    # LOVELAND = 391      
    # MAINEVILLE = 939
    # MASON = 935
    # MILDOFR = 127
    # MONROE = 89
    # MT HEALTHY = 395
    # READING = 399
    # ROSS C = 788
    # SHARONVILLE = 401
    # DAYTON = 661
    # HUBER HEIGHTS = 670
    # Trotwood = 668

    time.sleep(15)

    search_button = sb.driver.find_element("xpath", '//*[@id="btnSearch"]')
    search_button.click()

    # ---------- request download pdf file

    cookies = sb.driver.get_cookies()
    
    # Inicializa a variável para armazenar o HTML
    full_html = ""

    # while True:
        # Obtém o HTML da página atual e concatena
    html = sb.get_page_source()
    full_html += html

        

        # # Tenta encontrar o botão de "Próximo"
        # try:
        #     next_button = sb.driver.find_element("xpath", "/html/body/div[2]/div/div/div[2]/div[2]/div/div[3]/div/ul/li[3]/a")
        #     if next_button.get_attribute("class"):
        #         break  # Sai do loop se o botão estiver desabilitado
        #     next_button.click()
        #     time.sleep(3)  # Espera a próxima página carregar
            
        # except Exception as e:
        #     print("Não foi possível encontrar o botão de Próximo ou ocorreu um erro:", e)
        #     break


In [5]:
# Dicionário para armazenar os valores dos cookies
cookies_dict = {}

# Itera sobre a lista de cookies e extrai os valores
for cookie in cookies:
    if cookie['name'] == '__RequestVerificationToken':
        cookies_dict['__RequestVerificationToken'] = cookie['value']
    elif cookie['name'] == 'ASP.NET_SessionId':
        cookies_dict['ASP.NET_SessionId'] = cookie['value']

# Exibe os valores extraídos
print("Valores dos cookies:")
print("__RequestVerificationToken:", cookies_dict.get('__RequestVerificationToken'))
print("ASP.NET_SessionId:", cookies_dict.get('ASP.NET_SessionId'))

request_cookie = cookies_dict.get('__RequestVerificationToken')
ASP = cookies_dict.get('ASP.NET_SessionId')


from bs4 import BeautifulSoup

formID=[]
btnID=[]
tokens=[]

soup = BeautifulSoup(full_html, 'html.parser')
if 'No results found matching your criteria' in soup.text:
    rowns_table_quant=0
else:
    rowns_table = soup.find("table", class_="table table-primary table-vertical-center table-hover table-bordered table-striped").find_all("tr")
    rowns_table_quant = (len(rowns_table)-1)
    # Itera sobre as linhas da tabela (ignorando o cabeçalho, se houver)
    for row in rowns_table[1:]:  # Começa da segunda linha para ignorar o cabeçalho
        # Verifica se há um formulário na linha
        form = row.find('form')
        if form:
            form_id = form.get('id')
            formID.append(form_id)

            # Busca o token de verificação dentro do formulário
            token_input = form.find('input', {'name': '__RequestVerificationToken'})
            if token_input:
                token_value = token_input.get('value')
                tokens.append(token_value)
            else:
                tokens.append(None)  # Adiciona None se não houver token
        else:
            formID.append(None)  # Adiciona None se não houver formulário
            tokens.append(None)  # Adiciona None se não houver formulário

        # Verifica se há um botão na linha
        button = row.find('button')
        if button:
            button_id = button.get('id')
            btnID.append(button_id)
        else:
            btnID.append(None)  # Adiciona None se não houver botão

print(formID)
print(btnID)
print(tokens)




Valores dos cookies:
__RequestVerificationToken: jrX0FvQn7MoDZe6OhAul4rfsic1GD0CLZEddEbQezNrfgRt-5WRDBLU2oRRuKPOJtu5w0-xuexqvR0IUN-AgzdcznyJiNfvo1q8kSmKKJZk1
ASP.NET_SessionId: v24lnehtbxwp25gs4rt4tgf4
['vD0G507022j20226vuuP', 'vD0G517012j20296vuuP', 'vD0G597092j20216vuuP', 'vD0G517002j20296vuuP', 'vD0G517052j20216vuuP', 'vD0G587042j20256vuuP', 'vD0G567052j20206vuuP', 'vD0G537032j20256vuuP', 'vD0G507092j20296vuuP']
['btn_vD0G507022j20226vuuP', 'btn_vD0G517012j20296vuuP', 'btn_vD0G597092j20216vuuP', 'btn_vD0G517002j20296vuuP', 'btn_vD0G517052j20216vuuP', 'btn_vD0G587042j20256vuuP', 'btn_vD0G567052j20206vuuP', 'btn_vD0G537032j20256vuuP', 'btn_vD0G507092j20296vuuP']
['oGF7ZXFKXwfwTNDst11GPOskzftXSmBGSInNE0xCl8RJ35w2_u-W27cvQ4gsmTr710BRHEfEJCIc5yKyWt5wUEJcsWORHPxtgeVdREi40iA1', 'HJa6ROcqSFmjA7rao-d3fsJ5mHKFl6ZBhlh4FTdlcE6DUDezg0Hc0SMGYVjyUhxfXeNATacZ56P_p6_BZyf2tAIzwOtmqdRKqAqK4BkGUAs1', 'lJK6Pqp5UZlks3bijGWAKLEZeHCpDkt4i3z6DtoAM44cK_R8JajDHCB1oLG23cbsYcSIRx72MebsbL_VyczZDYUNV3i0ahd54nZrXY

In [6]:
import pdfplumber
import os
import re
import pandas as pd

username = 'rrel8ed'
password = 'zFfUPRWH6q'
country = 'US'
entry = ('http://customer-%s-cc-%s:%s@pr.oxylabs.io:7777' %
    (username, country, password))

proxies = {
    'http': entry,
    'https': entry,
}

df_final=pd.DataFrame(columns=["TYPE","Accident Report Number","Crash Date", "Unit","Unit at Fault","Name","Birth","Age","Gender","Address","License Plate","Insurance Policy","Insurance Company"])
for i in range(len(formID)):

    report_number=''
    crash_datetime=''
    dict_cars={}
    list_people=[]
    # print(formID)

    cookies = {
            'ASP.NET_SessionId': ASP,
            '__RequestVerificationToken': request_cookie,
        }
    
    headers = {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
        'Accept-Language': 'en-US,en;q=0.9,pt;q=0.8',
        'Cache-Control': 'max-age=0',
        'Connection': 'keep-alive',
        'Content-Type': 'application/x-www-form-urlencoded',
        'Origin': 'https://ohtrafficdata.dps.ohio.gov',
        'Referer': 'https://ohtrafficdata.dps.ohio.gov/CrashRetrieval/OhioCrashReportRetrieval/Search',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-User': '?1',
        'Upgrade-Insecure-Requests': '1',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36',
        'sec-ch-ua': '"Not(A:Brand";v="99", "Google Chrome";v="133", "Chromium";v="133"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"Windows"',
        # 'Cookie': 'ASP.NET_SessionId=41q4gzhcgxjpjzawxnkqrogx; __RequestVerificationToken=O4ItidAPXZX59vGrGn1YYhEYsMb_WRHr8S1vV5-p16s_qj2kBcjxWO8KTV3PBirwA97UVlsfqeZTVSEG69FqB69bSs4nHNSdE9w0lSNpYMs1',
    }

    data = {
        '__RequestVerificationToken': tokens[i],
        'id': formID[i],
        btnID[i]: '',
    }
    # print(formID[i])
   
    chunk_size=2000
    response = requests.post(
        'https://ohtrafficdata.dps.ohio.gov/CrashRetrieval/OhioCrashReportRetrieval/GetReport',
        cookies=cookies,
        headers=headers,
        data=data,
        proxies=proxies,
        stream=True
    )
    with open(os.path.join(path_data,"'output.pdf'"), 'wb') as fd:
        for chunk in response.iter_content(chunk_size):
            fd.write(chunk)
        
    with pdfplumber.open(os.path.join(path_data,"'output.pdf'")) as pdf:
        texto = ''
        # Extrair texto de cada página
        for pagina_count in range(len(pdf.pages)):
            partial_text = pdf.pages[pagina_count].extract_text()
            # print(partial_text)

            if '*DENOTES MANDATORY FIELD FOR SUPPLEMENT REPORT LOCAL REPORT NUMB' in partial_text.upper():
                 # Capturar o número do relatório
                report_match = re.search(r'LOCAL INFORMATION\s+.+[A-Za-z0-9-]+', partial_text)
                report_match=re.findall('[A-Za-z0-9-]*\d[A-Za-z0-9-]*',report_match[0]) if report_match else []
                report_number=report_match[0] if len(report_match)>0 else 'N/A'
                
                # Capturar a data e hora mais próxima do número do relatório
                datas_horas = re.finditer(r'(\d{2}/\d{2}/\d{4} \d{2}:\d{2})', partial_text)
                crash_datetime = "N/A"
                menor_distancia = float('inf')

                # Capturar Unit in Error (antes da sequência 9 98 9)
                unit_in_error_match = re.search(r'(\d+)\s+9\s+98\s+9\s+-\s+-', partial_text)
                unit_in_error = unit_in_error_match.group(1) if unit_in_error_match else 'N/A'

                report_match = re.search(r'LOCAL INFORMATION\s+(\S+)', partial_text)
                if report_match:
                    posicao_report = report_match.start()
                    for match in datas_horas:
                        posicao_data = match.start()
                        distancia = abs(posicao_data - posicao_report)
                        if distancia < menor_distancia:
                            menor_distancia = distancia
                            crash_datetime = match.group(1)

            elif 'UNIT' in partial_text.upper() and 'OWNER NAME:' in partial_text.upper():
                #in order to match car with person, need to get the unit
                unit=partial_text.split('OWNER NAME:')[-1].split('\n')[1][:2].strip()
                license_plate=''
                policy=''
                year=''
                insurance_policy=''
                insurance_company=''

                # Expressão regular ajustada
                content_partial=re.findall('\n.*\n.*VEHICLE MODEL.*\n.*\n',partial_text)
                has_insurance_content='X VERIFIED' in str(content_partial) or 'XVERIFIED' in str(content_partial)
                if content_partial:
                    for content_subpartial in content_partial:
                        try:
                            if license_plate=='':
                                license_plate=content_subpartial.strip().split('\n')[0].split(' ')[1]
                                if license_plate.strip()=='STATE':
                                    license_plate=''
                                    continue
                            if has_insurance_content and insurance_policy=='':
                                policy=content_subpartial.strip().split('\n')[0].split(license_plate)[1].strip()
                                year=re.findall('\d\d\d\d',policy)[-1]
                                insurance_policy=year.join(policy.split(year)[0:-1]).strip()
                                policy=content_subpartial.split('X VERIFIED')[-1].strip()
                                insur_number=re.findall(' \w*\d\w* ',policy)[0] if re.findall(' \w*\d\w* ',policy) else ''
                                if insur_number:
                                    insurance_company=policy.strip().split(insur_number)[0].strip()
                                    insurance_policy=policy.split(insurance_company)[-1].split()[0].strip()

                            break
                        
                        except:
                             pass
                license_plate='N/A' if license_plate=='' else license_plate
                insurance_policy='N/A' if insurance_company=='' else insurance_policy #based on insurance_company
                insurance_company='N/A' if insurance_company=='' else insurance_company
                dict_cars[unit]={
                        'license_plate':license_plate,
                        'policy':policy,
                        'year':year,
                        'insurance_policy':insurance_policy,
                        'insurance_company':insurance_company
                    }

            elif ('OTORIST / ON- OTORIST' in partial_text.upper()) or ('ITNESS' in partial_text.upper() and 'DDENDUM' in partial_text.upper()):
                #get person type:
                person_type=""
                if ('OTORIST / ON- OTORIST' in partial_text.upper()):
                    person_type = "DRIVER"
                elif ('ITNESS' in partial_text.upper() and 'DDENDUM' in partial_text.upper()):
                    person_type = "OCCUPANT"
                # get the people involved and place in dictionary
                regex = r"UNIT # NAME: LAST, FIRST, MIDDLE DATE OF BIRTH AGE GENDER\n(\d)+\s+([^,]+),\s+([^,\n]+)(?:,\s+([^,\n]+))?\s+(\d{2}/\d{2}/\d{4})\s+(\d+)\s+([MF])\nADDRESS: STREET, CITY, STATE, ZIP CONTACT PHONE - INCLUDE AREA CODE\n([^\n]+)"
                matches = re.findall(regex, partial_text)
                for i, match in enumerate(matches):
                    unit, last_name, first_name, middle_name, dob, age, gender, address = match
                    list_people.append({       
                                        'unit':unit.strip(),
                                        'last_name':last_name.strip(),
                                        'first_name':first_name.strip(),
                                        'middle_name':middle_name.strip(),
                                        'dob':dob.strip(),
                                        'age':age.strip(),
                                        'gender':gender.strip(),
                                        'address':address.strip(),
                                        'person_type':person_type.strip(),                     
                                        })
        print(list_people)
        print(dict_cars)
        list_cars_found=[]
        # now with all information from pdf, lets append it to the final dataframe
        for person in list_people:
            #first initialize all output variables:
            license_plate=''
            insurance_policy=''
            insurance_policy=''

            unit = person['unit']
            person_type = person['person_type']
            first_name = person['first_name']
            middle_name = person['middle_name']
            last_name = person['last_name']
            dob = person['dob']
            age = person['age']
            gender = person['gender']
            address = person['address']

            #get car information if it exists
            if unit in dict_cars.keys():
                list_cars_found.append(unit)
                license_plate=dict_cars[unit]['license_plate']
                insurance_policy=dict_cars[unit]['insurance_policy']
                insurance_company=dict_cars[unit]['insurance_company']

            df_final.loc[len(df_final)]=[
                person_type,
                report_number,
                crash_datetime,
                unit,
                unit_in_error,
                f"{first_name.strip()} {middle_name.strip()} {last_name.strip()}",
                dob.strip(),
                int(age.strip()),
                gender.strip(),
                address.strip(),
                license_plate, 
                insurance_policy,  
                insurance_company,
            ]
        for unit in dict_cars.keys(): #for cases where the car was not matched with anybody
            if not unit in list_cars_found:
                license_plate=dict_cars[unit]['license_plate']
                insurance_policy=dict_cars[unit]['insurance_policy']
                insurance_company=dict_cars[unit]['insurance_company']
                df_final.loc[len(df_final)]=[
                    '', #person_type
                    report_number,
                    crash_datetime,
                    unit,
                    unit_in_error,
                    '', #name 
                    '', #dob
                    '', #age
                    '', #gender
                    '', #address
                    license_plate, 
                    insurance_policy,  
                    insurance_company,
                ]

        # Criar uma cópia do arquivo como "xx.pdf"
        with open(os.path.join(path_data,"'output.pdf'"), "rb") as temp_pdf, open(os.path.join(path_data,report_number+".pdf"), "wb") as copy_pdf:
            copy_pdf.write(temp_pdf.read())


# Carregar os dados
df_corrigido = df_final

# Função auxiliar para aplicar no groupby
def preencher_policy_company(grupo):
    # Pegando qual unidade foi culpada
    unit_at_fault = grupo['Unit at Fault'].iloc[0]
    
    # Localiza a linha correspondente à unidade culpada
    culpado = grupo[grupo['Unit'] == unit_at_fault]
    
    if not culpado.empty:
        policy = culpado['Insurance Policy'].iloc[0]
        company = culpado['Insurance Company'].iloc[0]
    else:
        policy = None
        company = None

    # Adiciona novas colunas ao grupo todo
    grupo['Unit at Fault Policy'] = policy
    grupo['Unit at Fault Company'] = company

    return grupo

# Aplica por grupo de acidente
df_corrigido = df_corrigido.groupby('Accident Report Number', group_keys=False).apply(preencher_policy_company)

#add current date to file name
file_name=os.path.join(path_data,'Data_Crashes_Cincinnati_24H_'+past_day+'.csv')

#save file in saved_files/ folder
# file_name=os.path.join('saved_files',file_name)

df_corrigido.to_csv(file_name,index=False)
print("File loaded!")

[{'unit': '1', 'last_name': 'SMITH', 'first_name': 'TREVON', 'middle_name': '', 'dob': '06/11/1995', 'age': '29', 'gender': 'M', 'address': '505 KESTER AVE, DAYTON, OH, 45403', 'person_type': 'DRIVER'}, {'unit': '2', 'last_name': 'SHARP', 'first_name': 'DANIEL', 'middle_name': '', 'dob': '02/08/1965', 'age': '60', 'gender': 'M', 'address': '5951 LITTLE SUGAR CREEK RD, KETTERING, OH, 45440', 'person_type': 'DRIVER'}, {'unit': '1', 'last_name': 'SMITH', 'first_name': 'ROLAYSHA', 'middle_name': '', 'dob': '02/29/2000', 'age': '25', 'gender': 'F', 'address': '505 KESTER AVE, DAYTON, OH, 45403', 'person_type': 'OCCUPANT'}]
{'1': {'license_plate': 'JSR4772', 'policy': 'OHIO MUTUAL NSA7794727 GRN ELEMENT', 'year': '2004', 'insurance_policy': 'NSA7794727', 'insurance_company': 'OHIO MUTUAL'}, '2': {'license_plate': 'KFL7269', 'policy': 'PROGRESSIVE 930940083 GRN GRAND CHEROKEE', 'year': '2005', 'insurance_policy': '930940083', 'insurance_company': 'PROGRESSIVE'}}
[{'unit': '1', 'last_name': 'P

/var/folders/09/ybx6zt094qq75638ywz6q4q00000gn/T/ipykernel_74140/1314773507.py:264: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_corrigido = df_corrigido.groupby('Accident Report Number', group_keys=False).apply(preencher_policy_company)


File loaded!


In [15]:
df = pd.read_csv("/Users/cristianb/Documents/Python/rel8ed/SynapseIQ_Lab_01/Data/ohio/Data_Crashes_Cincinnati_24H_04.21.2025.csv")


In [16]:
insert_data_from_dataframe(df,"/home/data/ohio")